# ML Flow Tracking Server

In [1]:
import pandas as pd
import mlflow

In [4]:
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error

In [12]:
data_x=pd.read_csv('data/X_train.csv')
data_y=pd.read_csv('data/target.csv')


data=pd.concat([data_x, data_y], axis=1)
data.head()

,Married,Education,CoapplicantIncome,LoanAmount,Credit_History,Loan_Status
0,0.728816,-0.528362,1.369336,2.007739,4.516405e-01,1.0
1,0.728816,-0.528362,-0.202214,-0.552730,4.516405e-01,0.0
2,-1.372089,-0.528362,-0.554487,-0.790913,4.516405e-01,1.0
3,0.728816,-0.528362,-0.554487,-0.874277,4.516405e-01,1.0
4,-1.372089,-0.528362,0.466422,-0.171637,3.177548e-16,1.0


In [13]:
from mlflow.models import infer_signature
from urllib.parse import urlparse

In [14]:
data

,Married,Education,CoapplicantIncome,LoanAmount,Credit_History,Loan_Status
0,0.728816,-0.528362,1.369336,2.007739,4.516405e-01,1.0
1,0.728816,-0.528362,-0.202214,-0.552730,4.516405e-01,0.0
2,-1.372089,-0.528362,-0.554487,-0.790913,4.516405e-01,1.0
3,0.728816,-0.528362,-0.554487,-0.874277,4.516405e-01,1.0
4,-1.372089,-0.528362,0.466422,-0.171637,3.177548e-16,1.0
...,...,...,...,...,...,...
609,NaN,NaN,NaN,NaN,NaN,1.0
610,NaN,NaN,NaN,NaN,NaN,1.0
611,NaN,NaN,NaN,NaN,NaN,1.0
612,NaN,NaN,NaN,NaN,NaN,1.0


In [16]:
# Depndent & Independent

X=data.drop('Loan_Status', axis=1)
y=data['Loan_Status']


In [17]:
# Split the data to train & test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=40)

In [19]:
# Hyperparameter tuning GRidsearch cv

def hyperparameter_tuning(X_train, y_train, param_grid):
    rfc=RandomForestClassifier()
    grid_search=GridSearchCV(rfc, param_grid=param_grid, cv=3, n_jobs=-1, scoring='accuracy')

    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    return grid_search

In [25]:
signature=infer_signature(X_train, y_train)

# Define Hyperparamter tuning

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# Start ML Flow experiment 
mlflow.set_experiment('LoanEligibility')
with mlflow.start_run():
    #performing hyperparameter tuning
    grid_search=hyperparameter_tuning(X_train, y_train, param_grid)

    #best model
    best_model=grid_search.best_estimator_

    # Evaluate the model
    y_pred = best_model.predict(X_test)
    mse=mean_squared_error(y_test, y_pred)

    #Log best parameters & metrics
    mlflow.log_param('best n estimators', grid_search.best_params_['n_estimators'])
    mlflow.log_param('best max depth', grid_search.best_params_['max_depth'])
    mlflow.log_param('best min samples split', grid_search.best_params_['min_samples_split'])
    mlflow.log_param('best min samples leaf', grid_search.best_params_['min_samples_leaf'])
    mlflow.log_param('best max features', grid_search.best_params_['max_features'])
    mlflow.log_param('mse', mse)

    #tracking url
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

    tracking_url=urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url !='file':
        mlflow.sklearn.log_model(best_model, 'model', registered_model_name='Best Model')
    else:
        mlflow.sklearn.log_model(best_model, 'model', signature=signature, )

    print(f'Best parameters: {grid_search.best_params_}')
    print(f'Best mse: {mse}')



c:\Users\ntabjul\Downloads\ML Projects\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
324 fits failed out of a total of 972.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
43 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\ntabjul\Downloads\ML Projects\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\ntabjul\Downloads\ML Projects\.venv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
  File "c:\Users\ntabjul\Downloads\ML Projects\.venv\Lib\site-packages\sklearn\base.py", line 436, in _validate_params
    v

Best parameters: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
Best mse: 0.2926829268292683
🏃 View run stately-boar-569 at: http://127.0.0.1:5000/#/experiments/233442084668266576/runs/61fbd0fa06454a01b8e377475989f404
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/233442084668266576


Created version '2' of model 'Best Model'.
